In [12]:
from IPython.display import HTML, display

def enable_word_wrap():
    display(HTML("""
    <style>
        /* Force word wrap for all Jupyter output types */
        .jp-OutputArea-output, .output_subarea, .output_text, .output_pre, .jp-RenderedText {
            white-space: pre-wrap !important;
            word-break: break-all !important;
        }
    </style>
    """))

enable_word_wrap()


In [13]:
# Install the Anthropic SDK if needed
!pip install anthropic -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import anthropic

# ── Your API key ────────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
from anthropic import Anthropic

client  = Anthropic()

In [20]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages,model="claude-haiku-4-5", system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model":model,
        "max_tokens":1000,
        "messages":messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system 
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message.content[0].text

In [24]:
messages = []

add_user_message(messages,"Generate a very short event bridge rule as json")
add_assistant_message(messages,"'''json")

text=chat(messages, stop_sequences=["'''"])
text

'\n{\n  "Name": "MyRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n'

In [25]:
import json

json.loads(text.strip())

{'Name': 'MyRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1'}]}